**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# 🎯 Session 8: LangChain RAG 어플리케이션 구현

## 📋 학습 목표

1️⃣ LangChain의 Document Loaders로 다양한 형식의 문서를 로딩한다  
2️⃣ Text Splitters로 문서를 효과적으로 분할한다  
3️⃣ Embeddings와 VectorStore를 연동한다  
4️⃣ RetrievalQA 체인으로 RAG 어플리케이션을 구축한다  
5️⃣ 대화형 RAG (ConversationalRetrievalChain)를 구현한다  

---

### 🖥️ 실습 환경
- **GPU**: 불필요 (API 기반)
- **필수 패키지**: langchain, langchain-openai, langchain-community, chromadb
- **LLM**: OpenAI API 또는 Ollama 로컬 모델

## 1️⃣ 패키지 및 환경 설정

In [1]:
# 📦 패키지 확인 및 환경 설정
import importlib
import os

packages = [
    "langchain",
    "langchain_openai",
    "langchain_community",
    "chromadb",
]

print("📦 패키지 버전 확인")
print("=" * 40)
for pkg_name in packages:
    try:
        pkg = importlib.import_module(pkg_name)
        version = getattr(pkg, "__version__", "installed")
        print(f"  ✅ {pkg_name}: {version}")
    except ImportError:
        print(f"  ❌ {pkg_name}: 설치 필요")
        print(f"     pip install {pkg_name}")

📦 패키지 버전 확인
  ✅ langchain: 0.3.28


/home/yskim/LLM_Advanced/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  ✅ langchain_openai: installed
  ✅ langchain_community: 0.3.31
  ✅ chromadb: 1.5.7


In [2]:
# 🔑 API 키 설정
# 방법 1: 환경변수로 설정 (권장)
# os.environ["OPENAI_API_KEY"] = "sk-your-key-here"

# 방법 2: Ollama 로컬 모델 사용 (API 키 불필요)
USE_OLLAMA = True  # True면 Ollama, False면 OpenAI 사용

if USE_OLLAMA:
    print("🔧 Ollama 로컬 모델을 사용합니다.")
    print("   ollama serve 실행 후 진행해주세요.")
else:
    api_key = os.environ.get("OPENAI_API_KEY", "")
    if api_key:
        print(f"✅ OpenAI API 키 설정 완료 (키: ...{api_key[-4:]})")
    else:
        print("⚠️ OPENAI_API_KEY가 설정되지 않았습니다.")

🔧 Ollama 로컬 모델을 사용합니다.
   ollama serve 실행 후 진행해주세요.


## 2️⃣ LangChain Document Loaders

LangChain은 다양한 형식의 문서를 로딩하는 **Document Loader**를 제공합니다.

### 📁 주요 Document Loaders
| Loader | 지원 형식 | 설명 |
|--------|----------|------|
| `TextLoader` | .txt | 텍스트 파일 |
| `PyPDFLoader` | .pdf | PDF 파일 |
| `CSVLoader` | .csv | CSV 파일 |
| `WebBaseLoader` | URL | 웹 페이지 |
| `DirectoryLoader` | 폴더 | 폴더 내 전체 파일 |

In [3]:
# 📄 실습용 텍스트 파일 생성
import tempfile
import os

# 임시 디렉토리에 샘플 파일 생성
sample_dir = tempfile.mkdtemp(prefix="rag_docs_")

documents_content = {
    "ai_history.txt": """인공지능의 역사와 발전

인공지능(AI)은 1956년 다트머스 회의에서 처음 학문 분야로 정립되었습니다.
초기 AI는 규칙 기반 전문가 시스템이 주류였으며, 체스 프로그램 등이 대표적입니다.

1980년대에는 AI 겨울이 찾아와 연구 투자가 크게 줄었습니다.
하지만 2012년 AlexNet이 이미지넷 대회에서 압도적인 성적을 거두며 딥러닝 시대가 열렸습니다.

2017년 구글이 트랜스포머 아키텍처를 발표하면서 자연어 처리 분야에 혁명이 일어났습니다.
BERT, GPT 등의 대규모 언어 모델이 등장하여 다양한 NLP 태스크에서 인간 수준의 성능을 달성했습니다.

2022년 ChatGPT가 출시되며 AI가 일반 대중에게 널리 알려지게 되었습니다.
현재 생성형 AI는 텍스트, 이미지, 코드, 음악 등 다양한 콘텐츠를 생성할 수 있습니다.""",

    "llm_training.txt": """대규모 언어 모델(LLM) 학습 방법

사전학습(Pretraining)은 대량의 텍스트 데이터로 언어의 패턴을 학습하는 단계입니다.
다음 토큰 예측(Next Token Prediction) 방식으로 수조 개의 토큰을 학습합니다.

파인튜닝(Fine-tuning)은 특정 작업에 맞게 모델을 추가 학습하는 과정입니다.
SFT(Supervised Fine-Tuning)는 질문-답변 쌍으로 지시 따르기 능력을 향상시킵니다.

LoRA(Low-Rank Adaptation)는 전체 파라미터 중 일부만 학습하여 효율성을 높입니다.
QLoRA는 4bit 양자화와 LoRA를 결합하여 소비자급 GPU에서도 학습이 가능합니다.

RLHF(Reinforcement Learning from Human Feedback)는 인간의 선호도를 학습에 반영합니다.
DPO(Direct Preference Optimization)는 RLHF를 단순화한 방법론입니다.""",

    "rag_guide.txt": """RAG(Retrieval-Augmented Generation) 가이드

RAG는 검색 증강 생성이라고 하며, LLM의 환각 문제를 해결하는 핵심 기술입니다.
외부 지식 베이스에서 관련 정보를 검색한 후 LLM에 컨텍스트로 제공합니다.

RAG 파이프라인의 주요 단계:
1. 문서 수집 및 로딩
2. 텍스트 청킹 (적절한 크기로 분할)
3. 임베딩 생성 (텍스트를 벡터로 변환)
4. 벡터 DB에 저장 (ChromaDB, FAISS 등)
5. 사용자 질문에 대해 유사 문서 검색
6. 검색된 문서와 함께 LLM에 질의

청킹 전략, 임베딩 모델 선택, 검색 알고리즘이 RAG의 성능을 결정합니다.
Advanced RAG 기법으로 HyDE, Reranking, Query Expansion 등이 있습니다."""
}

for filename, content in documents_content.items():
    filepath = os.path.join(sample_dir, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print(f"📁 샘플 파일 생성 완료: {sample_dir}")
for f in os.listdir(sample_dir):
    size = os.path.getsize(os.path.join(sample_dir, f))
    print(f"  📄 {f} ({size} bytes)")

📁 샘플 파일 생성 완료: /tmp/rag_docs_0b8rzzvi
  📄 ai_history.txt (924 bytes)
  📄 llm_training.txt (844 bytes)
  📄 rag_guide.txt (739 bytes)


In [4]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader

# 📂 DirectoryLoader로 폴더 내 모든 텍스트 파일 로딩
print("📂 문서 로딩 시작")
print("=" * 50)

loader = DirectoryLoader(
    sample_dir,
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents = loader.load()

print(f"✅ {len(documents)}개 문서 로딩 완료")
for i, doc in enumerate(documents, 1):
    source = os.path.basename(doc.metadata.get('source', 'unknown'))
    print(f"  {i}. {source} ({len(doc.page_content)}자)")
    print(f"     미리보기: {doc.page_content[:80]}...")

📂 문서 로딩 시작
✅ 3개 문서 로딩 완료
  1. ai_history.txt (414자)
     미리보기: 인공지능의 역사와 발전

인공지능(AI)은 1956년 다트머스 회의에서 처음 학문 분야로 정립되었습니다.
초기 AI는 규칙 기반 전문가 시스템이...
  2. llm_training.txt (470자)
     미리보기: 대규모 언어 모델(LLM) 학습 방법

사전학습(Pretraining)은 대량의 텍스트 데이터로 언어의 패턴을 학습하는 단계입니다.
다음 토큰 ...
  3. rag_guide.txt (393자)
     미리보기: RAG(Retrieval-Augmented Generation) 가이드

RAG는 검색 증강 생성이라고 하며, LLM의 환각 문제를 해결하는 핵...


## 3️⃣ Text Splitters로 문서 분할

LangChain의 다양한 Text Splitter를 활용하여 문서를 청킹합니다.

In [5]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ✂️ RecursiveCharacterTextSplitter 사용
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)

print("✂️ 문서 분할 (RecursiveCharacterTextSplitter)")
print(f"  📏 chunk_size: 300, chunk_overlap: 50")
print("=" * 50)

splits = text_splitter.split_documents(documents)

print(f"\n📊 결과:")
print(f"  원본 문서: {len(documents)}개")
print(f"  분할 청크: {len(splits)}개")

# 청크 미리보기
print(f"\n🔍 처음 5개 청크 미리보기:")
for i, split in enumerate(splits[:5]):
    source = os.path.basename(split.metadata.get('source', 'unknown'))
    print(f"  {i+1}. [{source}] ({len(split.page_content)}자)")
    print(f"     {split.page_content[:80]}...")

✂️ 문서 분할 (RecursiveCharacterTextSplitter)
  📏 chunk_size: 300, chunk_overlap: 50

📊 결과:
  원본 문서: 3개
  분할 청크: 6개

🔍 처음 5개 청크 미리보기:
  1. [ai_history.txt] (201자)
     인공지능의 역사와 발전

인공지능(AI)은 1956년 다트머스 회의에서 처음 학문 분야로 정립되었습니다.
초기 AI는 규칙 기반 전문가 시스템이...
  2. [ai_history.txt] (211자)
     2017년 구글이 트랜스포머 아키텍처를 발표하면서 자연어 처리 분야에 혁명이 일어났습니다.
BERT, GPT 등의 대규모 언어 모델이 등장하여 ...
  3. [llm_training.txt] (234자)
     대규모 언어 모델(LLM) 학습 방법

사전학습(Pretraining)은 대량의 텍스트 데이터로 언어의 패턴을 학습하는 단계입니다.
다음 토큰 ...
  4. [llm_training.txt] (234자)
     LoRA(Low-Rank Adaptation)는 전체 파라미터 중 일부만 학습하여 효율성을 높입니다.
QLoRA는 4bit 양자화와 LoRA를 ...
  5. [rag_guide.txt] (289자)
     RAG(Retrieval-Augmented Generation) 가이드

RAG는 검색 증강 생성이라고 하며, LLM의 환각 문제를 해결하는 핵...


## 4️⃣ Embeddings & VectorStore 연동

LangChain의 Embeddings 인터페이스와 ChromaDB를 연동합니다.

In [6]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 📐 임베딩 모델 설정
print("📐 임베딩 모델 로딩 중...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)
print("✅ 임베딩 모델 로딩 완료")

# 🗃️ ChromaDB VectorStore 생성
print("\n🗃️ VectorStore 생성 중...")
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="langchain_rag_demo",
)

print(f"✅ VectorStore 생성 완료!")
print(f"📊 저장된 문서 수: {vectorstore._collection.count()}")

📐 임베딩 모델 로딩 중...


/tmp/ipykernel_39100/266942352.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


✅ 임베딩 모델 로딩 완료

🗃️ VectorStore 생성 중...
✅ VectorStore 생성 완료!
📊 저장된 문서 수: 6


In [7]:
# 🔍 VectorStore 검색 테스트
print("🔍 VectorStore 유사도 검색 테스트")
print("=" * 60)

query = "LoRA 파인튜닝이란 무엇인가요?"
print(f"❓ 쿼리: '{query}'\n")

# 유사도 검색
results = vectorstore.similarity_search_with_score(query, k=3)

for i, (doc, score) in enumerate(results, 1):
    source = os.path.basename(doc.metadata.get('source', 'unknown'))
    print(f"📌 결과 {i} (유사도 점수: {score:.4f})")
    print(f"   출처: {source}")
    print(f"   내용: {doc.page_content[:120]}...")
    print()

🔍 VectorStore 유사도 검색 테스트
❓ 쿼리: 'LoRA 파인튜닝이란 무엇인가요?'

📌 결과 1 (유사도 점수: 1.3347)
   출처: llm_training.txt
   내용: LoRA(Low-Rank Adaptation)는 전체 파라미터 중 일부만 학습하여 효율성을 높입니다.
QLoRA는 4bit 양자화와 LoRA를 결합하여 소비자급 GPU에서도 학습이 가능합니다.

RLHF(Reinfo...

📌 결과 2 (유사도 점수: 1.3764)
   출처: rag_guide.txt
   내용: 청킹 전략, 임베딩 모델 선택, 검색 알고리즘이 RAG의 성능을 결정합니다.
Advanced RAG 기법으로 HyDE, Reranking, Query Expansion 등이 있습니다....

📌 결과 3 (유사도 점수: 1.4649)
   출처: rag_guide.txt
   내용: RAG(Retrieval-Augmented Generation) 가이드

RAG는 검색 증강 생성이라고 하며, LLM의 환각 문제를 해결하는 핵심 기술입니다.
외부 지식 베이스에서 관련 정보를 검색한 후 LLM에 컨...



## 5️⃣ LLM 설정 (OpenAI 또는 Ollama)

RAG 체인에서 사용할 LLM을 설정합니다.

In [8]:
# 🤖 LLM 설정
if USE_OLLAMA:
    from langchain_community.llms import Ollama
    
    llm = Ollama(
        model="qwen2.5:1.5b",
        temperature=0.3,
        num_ctx=4096,        # 컨텍스트 길이 제한 (기본 32K → 4K, 속도 대폭 향상)
    )
    print("✅ Ollama LLM 설정 완료 (모델: qwen2.5:1.5b, num_ctx=4096)")
else:
    from langchain_openai import ChatOpenAI
    
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0.3,
    )
    print("✅ OpenAI LLM 설정 완료 (모델: gpt-4o-mini)")

# 간단한 테스트
print("\n🔧 LLM 테스트 중...")
try:
    test_response = llm.invoke("안녕하세요. 간단히 인사해주세요.")
    response_text = test_response if isinstance(test_response, str) else test_response.content
    print(f"💬 응답: {response_text[:100]}")
except Exception as e:
    print(f"❌ LLM 연결 실패: {e}")

✅ Ollama LLM 설정 완료 (모델: qwen2.5:1.5b, num_ctx=4096)

🔧 LLM 테스트 중...


/tmp/ipykernel_39100/1961382981.py:5: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(


💬 응답: 안녕하세요! 어떻게 도와드릴까요?


## 6️⃣ RetrievalQA 체인 구축

LangChain의 **RetrievalQA**는 검색 + 생성을 하나의 체인으로 연결합니다.

### 🔗 체인 동작 흐름
```
사용자 질문 → Retriever(검색) → 관련 문서 추출 → 프롬프트 구성 → LLM 응답
```

In [9]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# 📝 RAG 프롬프트 템플릿 정의
rag_prompt_template = """다음 컨텍스트를 참고하여 질문에 답변해주세요.
컨텍스트에 없는 정보는 '제공된 문서에서 해당 정보를 찾을 수 없습니다.'라고 답변하세요.

컨텍스트:
{context}

질문: {question}

답변:"""

PROMPT = PromptTemplate(
    template=rag_prompt_template,
    input_variables=["context", "question"]
)

print("📝 RAG 프롬프트 템플릿:")
print(rag_prompt_template)

📝 RAG 프롬프트 템플릿:
다음 컨텍스트를 참고하여 질문에 답변해주세요.
컨텍스트에 없는 정보는 '제공된 문서에서 해당 정보를 찾을 수 없습니다.'라고 답변하세요.

컨텍스트:
{context}

질문: {question}

답변:


In [10]:
# 🔗 RetrievalQA 체인 생성
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},  # 상위 3개 문서 검색
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",  # 검색 결과를 하나로 합쳐서 전달
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT},
)

print("✅ RetrievalQA 체인 생성 완료")
print("  🔹 검색 방식: similarity")
print("  🔹 검색 문서 수: 3")
print("  🔹 체인 타입: stuff (모든 문서를 하나로)")

✅ RetrievalQA 체인 생성 완료
  🔹 검색 방식: similarity
  🔹 검색 문서 수: 3
  🔹 체인 타입: stuff (모든 문서를 하나로)


In [11]:
# 🚀 RAG 질의응답 테스트
import time

def ask_rag(question):
    """RAG 체인에 질문하고 결과를 출력합니다."""
    print(f"❓ 질문: {question}")
    print("-" * 60)
    
    start_time = time.time()
    result = qa_chain.invoke({"query": question})
    elapsed = time.time() - start_time
    
    print(f"💬 답변:\n{result['result']}")
    print(f"\n⏱️ 소요 시간: {elapsed:.2f}초")
    
    print(f"\n📚 참고 문서:")
    for i, doc in enumerate(result['source_documents'], 1):
        source = os.path.basename(doc.metadata.get('source', 'unknown'))
        print(f"  {i}. [{source}] {doc.page_content[:80]}...")
    
    print("=" * 60)
    return result

# 테스트 질문
print("🚀 RAG 질의응답 테스트")
print("=" * 60)

ask_rag("LoRA와 QLoRA의 차이점은 무엇인가요?")

🚀 RAG 질의응답 테스트
❓ 질문: LoRA와 QLoRA의 차이점은 무엇인가요?
------------------------------------------------------------
💬 답변:
LoRA(Low-Rank Adaptation)는 전체 파라미터 중 일부만 학습하여 효율성을 높입니다. 반면, QLoRA는 4bit 양자화와 LoRA를 결합하여 소비자급 GPU에서도 학습이 가능합니다.

⏱️ 소요 시간: 0.73초

📚 참고 문서:
  1. [llm_training.txt] LoRA(Low-Rank Adaptation)는 전체 파라미터 중 일부만 학습하여 효율성을 높입니다.
QLoRA는 4bit 양자화와 LoRA를 ...
  2. [rag_guide.txt] RAG(Retrieval-Augmented Generation) 가이드

RAG는 검색 증강 생성이라고 하며, LLM의 환각 문제를 해결하는 핵...
  3. [rag_guide.txt] 청킹 전략, 임베딩 모델 선택, 검색 알고리즘이 RAG의 성능을 결정합니다.
Advanced RAG 기법으로 HyDE, Reranking, Qu...


{'query': 'LoRA와 QLoRA의 차이점은 무엇인가요?',
 'result': 'LoRA(Low-Rank Adaptation)는 전체 파라미터 중 일부만 학습하여 효율성을 높입니다. 반면, QLoRA는 4bit 양자화와 LoRA를 결합하여 소비자급 GPU에서도 학습이 가능합니다.',
 'source_documents': [Document(metadata={'source': '/tmp/rag_docs_0b8rzzvi/llm_training.txt'}, page_content='LoRA(Low-Rank Adaptation)는 전체 파라미터 중 일부만 학습하여 효율성을 높입니다.\nQLoRA는 4bit 양자화와 LoRA를 결합하여 소비자급 GPU에서도 학습이 가능합니다.\n\nRLHF(Reinforcement Learning from Human Feedback)는 인간의 선호도를 학습에 반영합니다.\nDPO(Direct Preference Optimization)는 RLHF를 단순화한 방법론입니다.'),
  Document(metadata={'source': '/tmp/rag_docs_0b8rzzvi/rag_guide.txt'}, page_content='RAG(Retrieval-Augmented Generation) 가이드\n\nRAG는 검색 증강 생성이라고 하며, LLM의 환각 문제를 해결하는 핵심 기술입니다.\n외부 지식 베이스에서 관련 정보를 검색한 후 LLM에 컨텍스트로 제공합니다.\n\nRAG 파이프라인의 주요 단계:\n1. 문서 수집 및 로딩\n2. 텍스트 청킹 (적절한 크기로 분할)\n3. 임베딩 생성 (텍스트를 벡터로 변환)\n4. 벡터 DB에 저장 (ChromaDB, FAISS 등)\n5. 사용자 질문에 대해 유사 문서 검색\n6. 검색된 문서와 함께 LLM에 질의'),
  Document(metadata={'source': '/tmp/rag_docs_0b8rzzvi/rag_guide.txt'}, page_content='청킹 전략, 임베

In [12]:
# 📝 다양한 질문 테스트
questions = [
    "트랜스포머는 언제 발표되었나요?",
    "RAG 파이프라인의 주요 단계를 설명해주세요.",
    "딥러닝 시대는 어떻게 시작되었나요?",
    "양자 컴퓨터의 원리는 무엇인가요?",  # 문서에 없는 내용
]

print("📝 다양한 질문 테스트")
print("=" * 60)

for q in questions:
    print()
    ask_rag(q)

📝 다양한 질문 테스트

❓ 질문: 트랜스포머는 언제 발표되었나요?
------------------------------------------------------------
💬 답변:
제공된 문서에서 해당 정보를 찾을 수 없습니다.

⏱️ 소요 시간: 0.34초

📚 참고 문서:
  1. [rag_guide.txt] 청킹 전략, 임베딩 모델 선택, 검색 알고리즘이 RAG의 성능을 결정합니다.
Advanced RAG 기법으로 HyDE, Reranking, Qu...
  2. [rag_guide.txt] RAG(Retrieval-Augmented Generation) 가이드

RAG는 검색 증강 생성이라고 하며, LLM의 환각 문제를 해결하는 핵...
  3. [llm_training.txt] 대규모 언어 모델(LLM) 학습 방법

사전학습(Pretraining)은 대량의 텍스트 데이터로 언어의 패턴을 학습하는 단계입니다.
다음 토큰 ...

❓ 질문: RAG 파이프라인의 주요 단계를 설명해주세요.
------------------------------------------------------------
💬 답변:
RAG 파이프라인의 주요 단계는 다음과 같습니다:

1. **문서 수집 및 로딩**:
   - 다양한 출처에서 문서를 수집합니다.
   - 수집된 문서들을 파일로 저장하거나 데이터베이스에 로드합니다.

2. **텍스트 청킹 (적절한 크기로 분할)**:
   - 수집된 텍스트를 적절한 크기로 분할하여 처리합니다. 이는 LLM의 입력 크기를 고려해야 합니다.
   
3. **임베딩 생성**:
   - 문서를 벡터로 변환하는 단계입니다. 일반적으로는 Transformers 모델을 사용하여 임베딩을 생성합니다.

4. **벡터 DB에 저장 (ChromaDB, FAISS 등)**:
   - 생성된 임베딩을 ChromaDB나 FAISS 같은 벡터 데이터베이스에 저장합니다.
   
5. **사용자 질문에 대해 유사 문서 검색**:
   - 사용자가 제시한 질문

## 7️⃣ 대화형 RAG (Conversational RAG)

이전 대화 내용을 기억하면서 RAG를 수행하는 **대화형 RAG**를 구현합니다.

### 💡 일반 RAG vs 대화형 RAG
- 일반 RAG: 매번 독립적인 질문으로 처리
- 대화형 RAG: 이전 대화를 참고하여 "그것"과 같은 대명사 해석 가능

In [13]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

# 💾 대화 메모리 설정
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer",
)

# 🔗 대화형 RAG 체인 생성
conversational_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
)

print("✅ 대화형 RAG 체인 생성 완료")
print("  🔹 대화 기록 유지: ConversationBufferMemory")
print("  🔹 이전 맥락을 참고하여 검색 쿼리 생성")

✅ 대화형 RAG 체인 생성 완료
  🔹 대화 기록 유지: ConversationBufferMemory
  🔹 이전 맥락을 참고하여 검색 쿼리 생성


/tmp/ipykernel_39100/567851842.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


In [14]:
# 💬 대화형 RAG 테스트
def chat_rag(question):
    """대화형 RAG에 질문합니다."""
    print(f"👤 사용자: {question}")
    
    result = conversational_chain.invoke({"question": question})
    answer = result["answer"] if isinstance(result["answer"], str) else result["answer"].content
    
    print(f"🤖 AI: {answer}")
    print("-" * 60)
    return result

print("💬 대화형 RAG 테스트")
print("=" * 60)

# 대화 1: 초기 질문
chat_rag("파인튜닝 방법에는 어떤 것들이 있나요?")
print()

# 대화 2: 대명사 사용 (이전 맥락 참조)
chat_rag("그 중에서 메모리를 가장 적게 사용하는 방법은?")
print()

# 대화 3: 후속 질문
chat_rag("그 방법으로 학습하려면 어떤 GPU가 필요한가요?")
print()

💬 대화형 RAG 테스트
👤 사용자: 파인튜닝 방법에는 어떤 것들이 있나요?
🤖 AI: 파인튜닝 방법에는 훈련 데이터셋을 확장하거나 업데이트하는 것이 포함됩니다. 이는 새로운 데이터를 사용하여 모델이 더 잘 작동하도록 하거나, 기존 데이터에 대한 오류를 수정하는 데 도움이 됩니다.
------------------------------------------------------------

👤 사용자: 그 중에서 메모리를 가장 적게 사용하는 방법은?
🤖 AI: 파인튜닝 방법 중에서 메모리 사용량을 최소화하는 방법은 SFT(Supervised Fine-Tuning) 방식입니다. 이 방식에서는 질문-답변 쌍으로 지시 따르기 능력을 향상시키는데, 이를 통해 학습 데이터의 크기를 줄이면서도 모델 성능을 유지하거나 개선할 수 있습니다.
------------------------------------------------------------

👤 사용자: 그 방법으로 학습하려면 어떤 GPU가 필요한가요?
🤖 AI: 파인튜닝을 위해 필요한 GPU는 대규모 언어 모델(LLM) 학습에 적합한 고성능 그래픽 카드가 필요합니다. 이들은 특히 텍스트 데이터를 처리하는 데 중요한 역할을 합니다. 이러한 GPU의 주요 특징은:

1. **GPU 성능**: 1080 Ti 등급 이상의 성능이 요구됩니다.
2. **메모리 크기**: 16GB 이상의 메모리가 필요합니다.
3. **그래픽 카드 모델**: NVIDIA RTX 4090, A100, V100 같은 고성능 그래픽 카드를 사용해야 합니다.

이러한 장비는 파인튜닝 과정에서 매우 중요한 역할을 하며, 대규모 언어 모델의 학습 성능과 향상된 성능을 보장합니다.
------------------------------------------------------------



## 8️⃣ LCEL (LangChain Expression Language)로 RAG 구현

최신 LangChain에서는 **LCEL**을 사용하여 더 유연하게 RAG 체인을 구성할 수 있습니다.

In [15]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 📝 LCEL 기반 RAG 체인
def format_docs(docs):
    """검색된 문서를 하나의 문자열로 포맷합니다."""
    return "\n\n".join(doc.page_content for doc in docs)

rag_prompt = PromptTemplate(
    template="""다음 컨텍스트를 참고하여 질문에 한국어로 답변해주세요.

컨텍스트:
{context}

질문: {question}

답변:""",
    input_variables=["context", "question"]
)

# LCEL 체인 구성
# 참고: 아래 LCEL 체인은 순수 Python으로 쓰면 이렇게 됩니다:
#   docs = vectorstore.similarity_search(question, k=3)
#   context = "\n\n".join(doc.page_content for doc in docs)
#   prompt = f"컨텍스트:\n{context}\n\n질문: {question}\n\n답변:"
#   answer = llm.invoke(prompt)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("✅ LCEL 기반 RAG 체인 생성 완료")
print("  📊 구조: Retriever → Format → Prompt → LLM → Output")

# LCEL 체인 테스트
print("\n🚀 LCEL RAG 테스트")
print("=" * 50)

question = "RLHF와 DPO는 어떤 차이가 있나요?"
print(f"❓ 질문: {question}")

try:
    answer = rag_chain.invoke(question)
    response_text = answer if isinstance(answer, str) else answer.content
    print(f"\n💬 답변:\n{response_text}")
except Exception as e:
    print(f"❌ 에러: {e}")

✅ LCEL 기반 RAG 체인 생성 완료
  📊 구조: Retriever → Format → Prompt → LLM → Output

🚀 LCEL RAG 테스트
❓ 질문: RLHF와 DPO는 어떤 차이가 있나요?

💬 답변:
RLHF(Relative Learning with Human Feedback)와 DPO(Direct Prompt Optimization)는 인공지능 시스템에서 사용되는 두 가지 학습 방법입니다. 이 두 기법은 모두 인간의 피드백을 이용하여 모델의 성능을 향상시키지만, 그 방식과 과정에는 차이가 있습니다.

RLHF는 "Relative Learning with Human Feedback"를 의미합니다. 이 방법에서는 인간의 피드백을 통해 모델의 학습 과정에 직접적인 영향을 미칩니다. 이를 위해 먼저 모델의 출력 결과를 평가하고, 그 결과에 따라 인간이 모델의 성능을 개선하는 방향으로 피드백을 주어 모델이 수정하도록 합니다. 이 방법은 빠르게 학습할 수 있는 장점이 있지만, 피드백을 제공하기 위해 많은 시간과 노력이 필요하며, 모델의 출력 결과에 대한 이해도가 중요합니다.

DPO(Direct Prompt Optimization)는 "Direct Prompt Optimization"를 의미합니다. 이 방법에서는 인간의 피드백을 직접적으로 사용하여 모델의 성능을 개선하는 방식입니다. 이를 위해 먼저 모델이 학습하고 있는 데이터셋에서 특정 문제점이나 패턴을 찾아내고, 그에 대한 해결책을 제안받습니다. 이 해결책은 바로 모델에 적용되어 모델의 성능을 향상시킵니다. DPO는 피드백을 제공하기 위해 필요한 시간과 노력이 적지만, 이 방법은 인간의 이해도와 판단력이 중요하지 않으며, 모델의 출력 결과에 대한 이해도가 필요합니다.

따라서 RLHF는 피드백을 직접적으로 적용하여 모델의 성능을 개선하는 데 중점을 두고 있으며, DPO는 피드백을 직접적인 해결책으로 제공하여 모델의 성능을 향상시키는 방식입니다. 이 두 방법은 각각 다른 장단점이 있으므로, 실

In [16]:
# 🧹 정리
import shutil

# 임시 파일 정리
if os.path.exists(sample_dir):
    shutil.rmtree(sample_dir)
    print(f"🧹 임시 디렉토리 삭제: {sample_dir}")

print("\n📌 오늘의 핵심 정리:")
print("  1️⃣ LangChain Document Loaders로 다양한 문서 형식을 로딩")
print("  2️⃣ Text Splitters로 문서를 적절한 크기로 분할")
print("  3️⃣ HuggingFace Embeddings + ChromaDB로 벡터 검색")
print("  4️⃣ RetrievalQA로 검색+생성을 하나의 체인으로 구현")
print("  5️⃣ 대화형 RAG로 이전 맥락을 유지하며 대화")
print("  6️⃣ LCEL로 더 유연한 RAG 체인 구성 가능")

🧹 임시 디렉토리 삭제: /tmp/rag_docs_0b8rzzvi

📌 오늘의 핵심 정리:
  1️⃣ LangChain Document Loaders로 다양한 문서 형식을 로딩
  2️⃣ Text Splitters로 문서를 적절한 크기로 분할
  3️⃣ HuggingFace Embeddings + ChromaDB로 벡터 검색
  4️⃣ RetrievalQA로 검색+생성을 하나의 체인으로 구현
  5️⃣ 대화형 RAG로 이전 맥락을 유지하며 대화
  6️⃣ LCEL로 더 유연한 RAG 체인 구성 가능


---

## 🎯 실습 과제

1️⃣ 자신만의 문서(PDF 또는 TXT)를 로딩하여 RAG 시스템을 구축해보세요  
2️⃣ 프롬프트 템플릿을 수정하여 응답 품질을 개선해보세요  
3️⃣ `search_kwargs`의 `k` 값을 1, 3, 5로 바꿔 결과를 비교해보세요  

---

## 📚 참고 자료
- [LangChain 공식 문서](https://python.langchain.com/docs/)
- [LangChain RAG 튜토리얼](https://python.langchain.com/docs/tutorials/rag/)
- [LCEL 가이드](https://python.langchain.com/docs/concepts/lcel/)